In [ ]:
"""
Delta Engine: Complex Systems Framework for Algorithmic Trading
Based on: Glattfelder, Houweling, Olsen (2025) - arxiv:2501.06032

README: Model artifact constraints & upload contract

Overview:
- Delta Engine is an ONNX-exportable trading model that consumes a windowed time series and emits a 2-element prediction per sample: [direction, confidence].

Artifact constraints (must be satisfied for upload):
- File format: ONNX (.onnx)
- ONNX opset: >= 12 (opset_version 12 recommended)
- Max file size: 500 MB (server default; configured via MAX_MODEL_FILE_SIZE)

Input contract:
- Input tensor name: "input"
- Input dtype: float32
- Input shape: [batch_size, sequence_length, num_features]
  - Default sequence_length used across project: 60
  - batch_size must be dynamic (export with dynamic batch axis)
  - num_features must match the `features` metadata provided at upload

Output contract:
- Output tensor name: "output"
- Output dtype: float32
- Output shape: [batch_size, 2]
  - output[:, 0] => direction (mapped to -1.0 (short) .. +1.0 (long))
  - output[:, 1] => confidence (0.0 .. 1.0)

Metadata & upload fields (multipart/form-data to POST /api/models/upload):
- Required form fields: 
  - file => ONNX file
  - name => model name (string)
  - version => semantic version (string)
  - algorithm => algorithm name (string)
  - target_asset => e.g., "BTC/USD" (string)
  - features => JSON array of feature names (stringified JSON)
- Optional fields: description, hyperparameters (stringified JSON)
- Server will compute checksum (MD5) and validate ONNX structure and file size

Validation checklist (before uploading):
1. Export ONNX using torch.onnx.export with:
   - input_names=['input'], output_names=['output']
   - dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
   - opset_version >= 12
2. Run onnx.checker.check_model on the saved file
3. Run a smoke test using onnxruntime.InferenceSession and compare outputs to PyTorch (np.testing.assert_allclose with rtol=1e-3)
4. Ensure `features` metadata length == num_features used by the model
5. Compute MD5 checksum and optionally include/verify it during upload

Post-processing & server-side expectations:
- If your model outputs a continuous price forecast, implement mapping to a `direction` and `confidence`:
  - direction: use tanh or sign mapping to -1..1
  - confidence: use sigmoid or normalized magnitude to [0,1]
- The real-time system consumes TradeSignal messages: model server should translate outputs into `trade_signal` messages with fields like `signalId`, `modelId`, `symbol`, `direction`, `confidence`, `entryPrice` and optional `stopLoss`/`takeProfit`.

Example curl upload (replace host and file paths):

curl -X POST http://your-api/api/models/upload \
  -F "file=@delta_engine.onnx" \
  -F "name=Delta Engine" \
  -F "version=1.0.0" \
  -F "algorithm=Delta Engine (Complex Systems)" \
  -F "target_asset=BTC/USD" \
  -F 'features=["close"]' \
  -F 'hyperparameters={"sequence_length":60, "num_features":1}'

Notes:
- Changing sequence_length or num_features requires retraining and re-exporting the ONNX model.
- The dashboard validates files on upload and forwards artifacts to the model VM for deployment; successful deployment updates model status to `active` in the database.

"""

import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from typing import List, Tuple
import json


class IntrinsicTimeProcessor(nn.Module):
    """
    Intrinsic Time: Directional Changes and Overshoots
    Implements the event-based temporal framework at multiple scales
    """
    
    def __init__(self, thresholds: List[float]):
        super().__init__()
        self.thresholds = torch.tensor(thresholds, dtype=torch.float32)
        
    def forward(self, prices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            prices: [batch_size, seq_len] - price time series
        Returns:
            features: [batch_size, n_thresholds * 4] - intrinsic time features
        """
        batch_size, seq_len = prices.shape
        features_list = []
        
        for threshold in self.thresholds:
            # Compute returns
            returns = (prices[:, 1:] - prices[:, :-1]) / (prices[:, :-1] + 1e-8)
            
            # Detect directional changes (DC count as volatility proxy)
            cumulative_change = torch.cumsum(returns, dim=1)
            dc_events = (torch.abs(cumulative_change) > threshold).float()
            dc_count = torch.sum(dc_events, dim=1, keepdim=True)
            
            # Compute overshoot characteristics
            overshoot_length = torch.mean(torch.abs(returns), dim=1, keepdim=True)
            overshoot_variance = torch.var(returns, dim=1, keepdim=True)
            
            # Volatility proxy (normalized DC count) - scaling law
            volatility = dc_count / (seq_len * threshold + 1e-8)
            
            features_list.append(torch.cat([
                dc_count / seq_len,  # Normalized DC count
                overshoot_length,
                overshoot_variance,
                volatility
            ], dim=1))
        
        return torch.cat(features_list, dim=1)


class SupportResistanceDetector(nn.Module):
    """
    Fits support and resistance lines to overshoot events
    Uses linear regression approximation via neural layers
    """
    
    def __init__(self, lookback_sizes: List[int]):
        super().__init__()
        self.lookback_sizes = lookback_sizes
        
    def forward(self, prices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            prices: [batch_size, seq_len]
        Returns:
            features: [batch_size, n_lookbacks * 5]
        """
        batch_size, seq_len = prices.shape
        features_list = []
        
        for lookback in self.lookback_sizes:
            window = min(lookback, seq_len)
            recent_prices = prices[:, -window:]
            
            # Local extrema
            local_max = torch.max(recent_prices, dim=1, keepdim=True)[0]
            local_min = torch.min(recent_prices, dim=1, keepdim=True)[0]
            current_price = prices[:, -1:]
            
            # Resistance line: distance to upper bound
            resistance_distance = (local_max - current_price) / (local_max - local_min + 1e-8)
            
            # Support line: distance to lower bound
            support_distance = (current_price - local_min) / (local_max - local_min + 1e-8)
            
            # Trend slope (linear fit approximation)
            price_change = recent_prices[:, -1:] - recent_prices[:, 0:1]
            trend_slope = price_change / (recent_prices[:, 0:1] + 1e-8)
            
            # Price position in range [0, 1]
            price_position = (current_price - local_min) / (local_max - local_min + 1e-8)
            
            # Trend strength (momentum)
            momentum = (prices[:, -1:] - prices[:, -window:].mean(dim=1, keepdim=True)) / (prices[:, -window:].std(dim=1, keepdim=True) + 1e-8)
            
            features_list.append(torch.cat([
                resistance_distance,
                support_distance,
                trend_slope,
                price_position,
                momentum
            ], dim=1))
        
        return torch.cat(features_list, dim=1)


class BreakoutSignalGenerator(nn.Module):
    """
    Generates contrarian breakout signals
    Neural implementation of pattern detection
    """
    
    def __init__(self, input_dim: int):
        super().__init__()
        self.signal_net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)  # [breakout_up, breakout_down, strength, volatility_adj]
        )
        
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.signal_net(features)


class MultiScaleAgentAggregator(nn.Module):
    """
    Aggregates signals from multi-scale agents
    Implements emergent collective decision-making
    """
    
    def __init__(self, n_agents: int):
        super().__init__()
        self.aggregator = nn.Sequential(
            nn.Linear(n_agents * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        
    def forward(self, agent_signals: List[torch.Tensor]) -> torch.Tensor:
        """
        Args:
            agent_signals: List of [batch_size, 4] tensors from each agent
        Returns:
            aggregated: [batch_size, 8]
        """
        combined = torch.cat(agent_signals, dim=1)
        return self.aggregator(combined)


class DeltaEngine(nn.Module):
    """
    Complete Delta Engine Implementation
    
    Architecture:
    1. Intrinsic Time Processing (multi-scale)
    2. Support/Resistance Detection
    3. Breakout Signal Generation
    4. Multi-Agent Aggregation
    5. Contrarian Pattern Detection
    6. Final Trading Signal
    
    Trading Logic:
    - Opens position of size u on initial signal
    - Reverses with 2u on contrarian breakout
    - Net exposure oscillates between +u and -u
    """
    
    def __init__(self,
                 delta_thresholds: List[float] = [0.0005, 0.001, 0.002, 0.005],
                 lookback_sizes: List[int] = [5, 10, 20, 30],
                 num_features: int = 1):
        super().__init__()
        
        self.delta_thresholds = delta_thresholds
        self.lookback_sizes = lookback_sizes
        self.num_features = num_features
        self.n_agents = len(delta_thresholds)
        
        # Core components
        self.intrinsic_time = IntrinsicTimeProcessor(delta_thresholds)
        self.support_resistance = SupportResistanceDetector(lookback_sizes)
        
        # Calculate feature dimensions
        intrinsic_dim = len(delta_thresholds) * 4
        sr_dim = len(lookback_sizes) * 5
        total_features = intrinsic_dim + sr_dim + num_features
        
        # Breakout signal generators (one per scale/agent)
        self.agents = nn.ModuleList([
            BreakoutSignalGenerator(total_features)
            for _ in range(self.n_agents)
        ])
        
        # Multi-scale aggregator
        self.aggregator = MultiScaleAgentAggregator(self.n_agents)
        
        # Final decision layer
        self.decision_head = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)  # [direction_logit, confidence_logit]
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass implementing the Delta Engine algorithm
        
        Args:
            x: [batch_size, 60, num_features] - input time series
               Feature 0 should be close price
        
        Returns:
            output: [batch_size, 2] where:
                output[:, 0] = direction (-1 for short, +1 for long)
                output[:, 1] = confidence (0 to 1)
        """
        batch_size, seq_len, n_features = x.shape
        
        # Extract close prices (assuming first feature)
        prices = x[:, :, 0]
        
        # 1. Intrinsic Time Features (multi-scale DC and overshoots)
        intrinsic_features = self.intrinsic_time(prices)
        
        # 2. Support/Resistance Features
        sr_features = self.support_resistance(prices)
        
        # 3. Combine with additional features
        last_features = x[:, -1, :]  # [batch_size, num_features]
        combined_features = torch.cat([
            intrinsic_features,
            sr_features,
            last_features
        ], dim=1)
        
        # 4. Generate signals from each agent (scale)
        agent_signals = []
        for agent in self.agents:
            signal = agent(combined_features)
            agent_signals.append(signal)
        
        # 5. Aggregate multi-scale signals (emergent behavior)
        aggregated = self.aggregator(agent_signals)
        
        # 6. Generate final trading signal
        decision = self.decision_head(aggregated)
        
        # Post-processing to match output contract
        # Direction: tanh to map to [-1, 1]
        direction = torch.tanh(decision[:, 0:1])
        
        # Confidence: sigmoid to map to [0, 1]
        confidence = torch.sigmoid(decision[:, 1:2])
        
        output = torch.cat([direction, confidence], dim=1)
        
        return output


def export_to_onnx(model: nn.Module,
                   output_path: str = "delta_engine.onnx",
                   seq_len: int = 60,
                   num_features: int = 1,
                   opset_version: int = 12) -> None:
    """
    Export Delta Engine to ONNX format with validation
    
    Args:
        model: Trained Delta Engine model
        output_path: Path to save ONNX file
        seq_len: Sequence length (default 60)
        num_features: Number of input features
        opset_version: ONNX opset version (default 12)
    """
    model.eval()
    
    # Create dummy input: [batch_size=1, seq_len=60, num_features]
    dummy_input = torch.randn(1, seq_len, num_features)
    
    # Export to ONNX
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=opset_version,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )
    
    print(f"✓ Model exported to {output_path}")
    
    # Validate ONNX model
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model validation passed")
    
    # Verify with ONNX Runtime
    ort_session = ort.InferenceSession(output_path)
    
    # Test inference
    ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.numpy()}
    ort_outputs = ort_session.run(None, ort_inputs)
    
    print(f"✓ ONNX Runtime inference successful")
    print(f"  Input shape: {ort_session.get_inputs()[0].shape}")
    print(f"  Output shape: {ort_session.get_outputs()[0].shape}")
    print(f"  Sample output: {ort_outputs[0][0]}")
    
    # Test with PyTorch
    with torch.no_grad():
        torch_output = model(dummy_input).numpy()
    
    # Compare outputs
    np.testing.assert_allclose(torch_output, ort_outputs[0], rtol=1e-3, atol=1e-5)
    print("✓ PyTorch and ONNX outputs match")
    
    return onnx_model


def prepare_metadata(model: DeltaEngine,
                     algorithm_name: str = "Delta Engine",
                     version: str = "1.0.0",
                     target_asset: str = "BTC/USD") -> dict:
    """
    Prepare metadata for model upload
    
    Returns:
        Dictionary with all required upload fields
    """
    # Feature names
    features = ["close"]  # Primary feature
    
    # Hyperparameters
    hyperparameters = {
        "delta_thresholds": model.delta_thresholds,
        "lookback_sizes": model.lookback_sizes,
        "sequence_length": 60,
        "num_features": model.num_features,
        "paradigm": "complex_systems",
        "intrinsic_time": True,
        "multi_scale": True,
        "agent_based": True,
        "contrarian_breakout": True,
        "volatility_aware": True
    }
    
    metadata = {
        "name": algorithm_name,
        "version": version,
        "algorithm": "Delta Engine (Complex Systems)",
        "target_asset": target_asset,
        "description": (
            "Delta Engine: Multi-scale agent-based trading algorithm using "
            "intrinsic time (directional changes & overshoots), support/resistance "
            "fitting, and contrarian breakout signals. Based on complex systems "
            "theory with emergent self-organized behavior."
        ),
        "features": features,
        "hyperparameters": json.dumps(hyperparameters),
        "input_shape": [None, 60, model.num_features],
        "output_shape": [None, 2],
        "output_description": {
            "0": "direction: -1 (short) to +1 (long)",
            "1": "confidence: 0 to 1"
        }
    }
    
    return metadata


# Example usage
if __name__ == "__main__":
    print("="*80)
    print("DELTA ENGINE: Complex Systems Framework for Algorithmic Trading")
    print("Based on: Glattfelder, Houweling, Olsen (2025) - arxiv:2501.06032")
    print("="*80)
    
    # Initialize model
    print("\n1. Initializing Delta Engine...")
    model = DeltaEngine(
        delta_thresholds=[0.0005, 0.001, 0.002, 0.005],  # Multi-scale thresholds
        lookback_sizes=[5, 10, 20, 30],  # Trend line fitting windows
        num_features=1  # Using only close price
    )
    
    print(f"   ✓ Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
    print(f"   ✓ Delta thresholds: {model.delta_thresholds}")
    print(f"   ✓ Lookback sizes: {model.lookback_sizes}")
    print(f"   ✓ Number of agents: {model.n_agents}")
    
    # Test forward pass
    print("\n2. Testing forward pass...")
    test_input = torch.randn(2, 60, 1)  # Batch of 2
    model.eval()
    with torch.no_grad():
        output = model(test_input)
    
    print(f"   ✓ Input shape: {test_input.shape}")
    print(f"   ✓ Output shape: {output.shape}")
    print(f"\n   Sample predictions:")
    for i in range(output.shape[0]):
        direction = "LONG" if output[i, 0] > 0 else "SHORT"
        print(f"   Sample {i+1}: Direction={direction}, "
              f"Signal={output[i, 0].item():.4f}, "
              f"Confidence={output[i, 1].item():.4f}")
    
    # Export to ONNX
    print("\n3. Exporting to ONNX...")
    onnx_model = export_to_onnx(
        model,
        output_path="delta_engine.onnx",
        seq_len=60,
        num_features=1,
        opset_version=12
    )
    
    # Prepare metadata
    print("\n4. Preparing upload metadata...")
    metadata = prepare_metadata(model, target_asset="BTC/USD")
    
    print("\n   Upload Contract:")
    print(f"   ✓ Name: {metadata['name']}")
    print(f"   ✓ Version: {metadata['version']}")
    print(f"   ✓ Algorithm: {metadata['algorithm']}")
    print(f"   ✓ Target Asset: {metadata['target_asset']}")
    print(f"   ✓ Features: {metadata['features']}")
    print(f"   ✓ Input shape: {metadata['input_shape']}")
    print(f"   ✓ Output shape: {metadata['output_shape']}")
    
    # Save metadata
    with open("delta_engine_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"\n   ✓ Metadata saved to delta_engine_metadata.json")
    
    print("\n" + "="*80)
    print("READY FOR UPLOAD")
    print("="*80)
    print("\nFiles generated:")
    print("  1. delta_engine.onnx - ONNX model file")
    print("  2. delta_engine_metadata.json - Upload metadata")
    print("\nUpload command example:")
    print("""
    curl -X POST http://your-api/api/models/upload \
      -F "file=@delta_engine.onnx" \
      -F "name=Delta Engine" \
      -F "version=1.0.0" \
      -F "algorithm=Delta Engine (Complex Systems)" \
      -F "target_asset=BTC/USD" \
      -F 'features=["close"]' \
      -F 'hyperparameters=@delta_engine_metadata.json'
    """)
    
    print("\n" + "="*80)
